In [1]:
import os
import boto3
import duckdb

In [2]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
print("Connected.")

Connected.


In [3]:
con.execute(open("/workspace/sql/30_gold.sql").read())
print("Gold tables created.")

Gold tables created.


In [4]:
result = con.sql("""
    SELECT clip_uri, fragment_id, start_frame, end_frame, n_objects, classes
    FROM silver.visdrone_fragments
    WHERE n_objects > 20
    ORDER BY n_objects DESC
    LIMIT 10
""").df()
print(result)

                                            clip_uri  fragment_id  \
0  s3://lakehouse/assets/visdrone/frames/uav00001...            4   
1  s3://lakehouse/assets/visdrone/frames/uav00001...            3   
2  s3://lakehouse/assets/visdrone/frames/uav00001...            0   
3  s3://lakehouse/assets/visdrone/frames/uav00001...            2   
4  s3://lakehouse/assets/visdrone/frames/uav00001...            1   
5  s3://lakehouse/assets/visdrone/frames/uav00001...            5   
6  s3://lakehouse/assets/visdrone/frames/uav00001...            6   
7  s3://lakehouse/assets/visdrone/frames/uav00001...           10   
8  s3://lakehouse/assets/visdrone/frames/uav00001...            9   
9  s3://lakehouse/assets/visdrone/frames/uav00001...            8   

   start_frame  end_frame  n_objects  \
0          121        150       3366   
1           91        120       3360   
2            1         30       3248   
3           61         90       3177   
4           31         60       3143   


In [5]:
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

for _, row in result.iterrows():
    prefix = row["clip_uri"].replace("s3://lakehouse/", "")
    for frame_id in range(row["start_frame"], row["end_frame"] + 1):
        key = f"{prefix}{str(frame_id).zfill(7)}.jpg"
        try:
            s3.head_object(Bucket="lakehouse", Key=key)
            print(f"EXISTS: {key}")
        except:
            print(f"MISSING: {key}")
    break  # just verify first fragment to keep output short

EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000121.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000122.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000123.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000124.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000125.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000126.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000127.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000128.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000129.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000130.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000131.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000132.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000133.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000134.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000135.jpg
EXISTS: assets/visdrone/frames/uav0000137_00458_v/0000136.jpg
EXISTS: 

In [6]:
con.sql("""
    SELECT image_uri, COUNT(*) AS n_people
    FROM silver.coco_annotations
    WHERE category = 'person'
    GROUP BY image_uri
    HAVING COUNT(*) >= 5
    ORDER BY n_people DESC
    LIMIT 10
""").df()

,image_uri,n_people
0,s3://lakehouse/assets/coco/images/85682.jpg,14
1,s3://lakehouse/assets/coco/images/74092.jpg,14
2,s3://lakehouse/assets/coco/images/236426.jpg,14
3,s3://lakehouse/assets/coco/images/141671.jpg,14
4,s3://lakehouse/assets/coco/images/60886.jpg,14
5,s3://lakehouse/assets/coco/images/78565.jpg,14
6,s3://lakehouse/assets/coco/images/229849.jpg,14
7,s3://lakehouse/assets/coco/images/76468.jpg,14
8,s3://lakehouse/assets/coco/images/188465.jpg,14
9,s3://lakehouse/assets/coco/images/507037.jpg,14


In [7]:
print(con.sql("FROM ducklake_snapshots('lake')").df())

    snapshot_id                    snapshot_time  schema_version  \
0             0 2026-06-25 19:47:58.592626+00:00               0   
1             1 2026-06-25 19:47:58.638004+00:00               1   
2             2 2026-06-25 19:47:58.647187+00:00               2   
3             3 2026-06-25 19:47:58.669349+00:00               3   
4             4 2026-06-25 22:18:44.460503+00:00               4   
5             5 2026-06-25 23:13:50.427815+00:00               5   
6             6 2026-06-25 23:13:50.502705+00:00               6   
7             7 2026-06-26 00:31:41.238716+00:00               7   
8             8 2026-06-26 00:31:56.698735+00:00               8   
9             9 2026-06-26 00:32:06.396628+00:00               9   
10           10 2026-06-26 01:47:33.156378+00:00              10   
11           11 2026-06-26 01:47:40.993846+00:00              11   
12           12 2026-06-26 01:57:16.493581+00:00              12   
13           13 2026-06-26 01:57:16.512439+00:00

In [8]:
con.close()